In [ ]:
from nrem_analysis.constant import PROCESSED_DIR, INTERIM_DIR
from pathlib import Path

import numpy as np
import pynapple as nap
import matplotlib.pyplot as plt

from cmap import Colormap
from matplotlib.colors import TwoSlopeNorm

cm = Colormap("colorbrewer:Set2")
norm = TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)

plt.rcParams.update({
    "figure.dpi":        150,
    "savefig.dpi":       320,
    "font.family":       "sans-serif",
    'font.sans-serif':   "Arial",
    "axes.grid":         False,
    "figure.constrained_layout.use": True,
})

AWAKE_STA_KWARGS = dict(binsize=0.05, window=(-2.025, 2.025))
NREM_STA_KWARGS = dict(binsize=0.005, window=(-0.4, 0.4))
MOUSE_IDS = ["99b", "103c", "106b", "107b", "110b"]

def compute_unwrapped_sta(
    data,
    events,
    window,
    binsize,
    time_unit="s",
):
    perievent = nap.compute_perievent(data=data, events=events, window=window, time_unit=time_unit,)
    # unwrap per each event
    unwrapped = np.unwrap(perievent, axis=0)
    mean_trace = np.nanmean(unwrapped, axis=1)
    # remove baseline
    zero_idx = np.argmin(np.abs(result.index))
    result = result - result[zero_idx]
    return result.bin_average(bin_size=binsize, time_units=time_unit)

def compute_unwrapped_stas(
    data,
    units,
    label,
    **kwargs,
):
    unit_stas = []

    for uid in units.index:
        events = units[uid].restrict(data.time_support)

        if len(events) == 0:
            print(f"Skipping unit {uid} for {label}")
            continue

        unit_stas.append(compute_unwrapped_sta(data=data, events=events, **kwargs,))

    return np.column_stack(unit_stas)

In [ ]:
tsi = []
sta_awake = []

for mouse_id in MOUSE_IDS:
    turn_units      = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "turn_units.npz")
    head_direction  = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "head_direction.npz")

    sta_awake.append(compute_unwrapped_stas(np.deg2rad(head_direction), turn_units, **AWAKE_STA_KWARGS))
    tsi.append(turn_units['turn_index'])

tsi = np.concatenate(tsi)
order = np.argsort(tsi)
sta_awake = np.column_stack(sta_awake)

In [ ]:
sta = sta_awake
plt.imshow(
    sta[:, order].d.T,
    aspect='auto',
    origin='lower',
    extent=[sta.index[0], sta.index[-1], 0, sta.shape[1]],
    cmap='hot',
    # norm=norm,
    )